env path:   
mamba activate /Volumes/nimas_usb/Docs/envs/ssdbase


In [1]:
import os
import re
import pandas as pd
from pathlib import Path

AvanaRawReadcounts.csv:  Raw sgRNA read counts for all screens and pDNA baselines.  
AvanaGuideMap.csv:       Mapping between sgRNAs and their target genes.  
ScreenSequenceMap.csv:   Metadata for replicates: cell line, QC status, pDNA indicator.

In [2]:
# Mac Data Paths
raw   = pd.read_csv(os.path.expanduser("~/Google Drive/My Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/data_raw/AvanaRawReadcounts.csv"))
guide = pd.read_csv(os.path.expanduser("~/Google Drive/My Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/data_raw/AvanaGuideMap.csv"))
seqmap = pd.read_csv(os.path.expanduser("~/Google Drive/My Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/data_raw/ScreenSequenceMap.csv"))

In [ ]:
# Linux Data Paths#
raw   = pd.read_csv(os.path.expanduser("~/Google Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/data_raw/AvanaRawReadcounts.csv"))
guide = pd.read_csv(os.path.expanduser("~/Google Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/data_raw/AvanaGuideMap.csv"))
seqmap = pd.read_csv(os.path.expanduser("~/Google Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/data_raw/ScreenSequenceMap.csv"))

### Exploratory codes

In [3]:
raw.shape, guide.shape, seqmap.shape

((74687, 2285), (170429, 6), (3238, 12))

In [4]:
"""
raw.columns[-10:].tolist()
raw.columns[raw.columns.str.startswith('pDNA_batch')].tolist()
with open(os.path.expanduser("~/Google Drive/My Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/notebooks/raw_columns.txt"), "w") as f:
    for col in raw.columns.tolist():
        f.write(col + "\n")
"""

'\nraw.columns[-10:].tolist()\nraw.columns[raw.columns.str.startswith(\'pDNA_batch\')].tolist()\nwith open(os.path.expanduser("~/Google Drive/My Drive/informatics/AI projects/crispr_ai_ml/crispr-ml-hitcalling/notebooks/raw_columns.txt"), "w") as f:\n    for col in raw.columns.tolist():\n        f.write(col + "\n")\n'

In [5]:
raw.head(3)

,Unnamed: 0,Colo699-311cas9-RepA-p6_Avana-4,Colo699-311cas9-RepB-p6_Avana-4,HSB-2-311cas9_RepA_p4_Avana-3,HSB-2-311cas9_RepB_p4_Avana-3,NCIH1339-311Cas9_RepA_p6_Avana-3,NCIH1339-311Cas9_RepB_p6_Avana-3,HEL-311Cas9_RepA_p4_Avana-3,HEL-311Cas9_RepB_p4_Avana-3,HEL9217-311Cas9_RepA_p6_Avana-3,...,HMMME-311cas9-RepB-p6_Avana-4,JVE528-311cas9-RepB-p6_Avana-4,UACC3133-311Cas9-RepB-P6_Avana-4,LU139-311cas9-RepB-p5_Avana-4,UPMM3-311cas9-RepA-p4_Avana-4,UACC3199-311cas9-RepA-p4_Avana-4,UACC3199-311cas9-RepB-p4_Avana-4,pDNA_batch_Avana-4,pDNA_batch_Avana-3,pDNA_batch_Avana-2
0,AAAAAAATCCAGCAATGCAG,463.0,521.0,514.0,776.0,474.0,329.0,214.0,254.0,510.0,...,450.0,471.0,460.0,419.0,275.0,528.0,546.0,2165.0,6327.0,190.5
1,AAAAAACCCGTAGATAGCCT,570.0,596.0,581.0,756.0,398.0,281.0,218.0,270.0,603.0,...,741.0,537.0,406.0,577.0,385.0,733.0,608.0,3127.5,9322.0,289.0
2,AAAAAAGAAGAAAAAACCAG,93.0,79.0,135.0,244.0,85.0,45.0,106.0,118.0,171.0,...,52.0,89.0,171.0,45.0,100.0,149.0,127.0,517.0,2030.5,41.5


In [6]:
raw.rename(columns={'Unnamed: 0': 'sgRNA_seq'}, inplace=True)
raw.head(3)

,sgRNA_seq,Colo699-311cas9-RepA-p6_Avana-4,Colo699-311cas9-RepB-p6_Avana-4,HSB-2-311cas9_RepA_p4_Avana-3,HSB-2-311cas9_RepB_p4_Avana-3,NCIH1339-311Cas9_RepA_p6_Avana-3,NCIH1339-311Cas9_RepB_p6_Avana-3,HEL-311Cas9_RepA_p4_Avana-3,HEL-311Cas9_RepB_p4_Avana-3,HEL9217-311Cas9_RepA_p6_Avana-3,...,HMMME-311cas9-RepB-p6_Avana-4,JVE528-311cas9-RepB-p6_Avana-4,UACC3133-311Cas9-RepB-P6_Avana-4,LU139-311cas9-RepB-p5_Avana-4,UPMM3-311cas9-RepA-p4_Avana-4,UACC3199-311cas9-RepA-p4_Avana-4,UACC3199-311cas9-RepB-p4_Avana-4,pDNA_batch_Avana-4,pDNA_batch_Avana-3,pDNA_batch_Avana-2
0,AAAAAAATCCAGCAATGCAG,463.0,521.0,514.0,776.0,474.0,329.0,214.0,254.0,510.0,...,450.0,471.0,460.0,419.0,275.0,528.0,546.0,2165.0,6327.0,190.5
1,AAAAAACCCGTAGATAGCCT,570.0,596.0,581.0,756.0,398.0,281.0,218.0,270.0,603.0,...,741.0,537.0,406.0,577.0,385.0,733.0,608.0,3127.5,9322.0,289.0
2,AAAAAAGAAGAAAAAACCAG,93.0,79.0,135.0,244.0,85.0,45.0,106.0,118.0,171.0,...,52.0,89.0,171.0,45.0,100.0,149.0,127.0,517.0,2030.5,41.5


In [7]:
print(raw.columns.to_list())

['sgRNA_seq', 'Colo699-311cas9-RepA-p6_Avana-4', 'Colo699-311cas9-RepB-p6_Avana-4', 'HSB-2-311cas9_RepA_p4_Avana-3', 'HSB-2-311cas9_RepB_p4_Avana-3', 'NCIH1339-311Cas9_RepA_p6_Avana-3', 'NCIH1339-311Cas9_RepB_p6_Avana-3', 'HEL-311Cas9_RepA_p4_Avana-3', 'HEL-311Cas9_RepB_p4_Avana-3', 'HEL9217-311Cas9_RepA_p6_Avana-3', 'HEL9217-311Cas9_RepB_p6_Avana-3', 'LS513-311Cas9_RepA_p6_Avana-2', 'LS513-311Cas9_RepB_p6_Avana-2', 'C2BBE1-311Cas9 Rep A p5_Avana-3', 'C2BBE1-311Cas9 Rep B p5_Avana-3', 'C2BBE1-311Cas9 Rep C p5_Avana-3', '253J-311Cas9_RepA_p5_Avana-3', 'HCC827-311Cas9 Rep A p6_Avana-3', 'HCC827-311Cas9 Rep B p6_Avana-3', 'HCC827-311Cas9 Rep C p6_Avana-3', 'ONCO-DG-1-311Cas9_RepA_p6_Avana-3', 'ONCO-DG-1-311Cas9_RepB_p6_Avana-3', 'Hs 294T_A101D-311Cas9_RepA_p6_Avana-3', 'Hs 294T_A101D-311Cas9_RepB_p6_Avana-3', 'NCI-H1581-311Cas9_RepA_p6_Avana-2', 'NCI-H1581-311Cas9_RepB_p6_Avana-2', 'SKBR3-311Cas9_RepA_p6_Avana-3', 'SKBR3-311Cas9_RepB_p6_Avana-3', 'T24-311Cas9_RepA_p6_Avana-3', 'T24-311Cas

In [8]:
guide.head(3)

,sgRNA,GenomeAlignment,nAlignments,Gene,DropReason,UsedByChronos
0,AAAAAAATCCAGCAATGCAG,chr10_110964620_+,1.0,SHOC2 (8036),NaN,True
1,AAAAAACCCGTAGATAGCCT,chr12_95003615_+,1.0,NDUFA12 (55967),NaN,True
2,AAAAAAGAAGAAAAAACCAG,chr4_75970356_-,1.0,SDAD1 (55153),NaN,True


In [9]:
guide['Gene'].nunique()

18531

In [10]:
seqmap.head()

,SequenceID,ScreenID,Days,pDNABatch,Replicate,ModelID,ExcludeFromCRISPRCombined,ScreenType,Library,ModelConditionID,DrugTreated,PassesQC
0,Colo699-311cas9-RepA-p6_Avana-4,SC-001041.AV01,21,Avana-4,A,ACH-001041,False,2DS,Avana,MC-001041-CkXW,False,True
1,Colo699-311cas9-RepB-p6_Avana-4,SC-001041.AV01,21,Avana-4,B,ACH-001041,False,2DS,Avana,MC-001041-CkXW,False,True
2,HSB-2-311cas9_RepA_p4_Avana-3,SC-001737.AV01,21,Avana-3,A,ACH-001737,False,2DS,Avana,MC-001737-6D5K,False,True
3,HSB-2-311cas9_RepB_p4_Avana-3,SC-001737.AV01,21,Avana-3,B,ACH-001737,False,2DS,Avana,MC-001737-6D5K,False,True
4,NCIH1339-311Cas9_RepA_p6_Avana-3,SC-000921.AV01,21,Avana-3,A,ACH-000921,False,2DS,Avana,MC-000921-P628,False,True


In [11]:
#seqmap['pDNABatch'].unique()
print(raw.iloc[:, -3:].head(3))
print("---------------------------------------------------------------------------------------------------------------------------------------")
#print(seqmap[seqmap['pDNABatch'].str.startswith('Avana', na=False)].head(3))
seqmap[seqmap['pDNABatch'].str.startswith('Avana', na=False)].head()


   pDNA_batch_Avana-4  pDNA_batch_Avana-3  pDNA_batch_Avana-2
0              2165.0              6327.0               190.5
1              3127.5              9322.0               289.0
2               517.0              2030.5                41.5
---------------------------------------------------------------------------------------------------------------------------------------


,SequenceID,ScreenID,Days,pDNABatch,Replicate,ModelID,ExcludeFromCRISPRCombined,ScreenType,Library,ModelConditionID,DrugTreated,PassesQC
0,Colo699-311cas9-RepA-p6_Avana-4,SC-001041.AV01,21,Avana-4,A,ACH-001041,False,2DS,Avana,MC-001041-CkXW,False,True
1,Colo699-311cas9-RepB-p6_Avana-4,SC-001041.AV01,21,Avana-4,B,ACH-001041,False,2DS,Avana,MC-001041-CkXW,False,True
2,HSB-2-311cas9_RepA_p4_Avana-3,SC-001737.AV01,21,Avana-3,A,ACH-001737,False,2DS,Avana,MC-001737-6D5K,False,True
3,HSB-2-311cas9_RepB_p4_Avana-3,SC-001737.AV01,21,Avana-3,B,ACH-001737,False,2DS,Avana,MC-001737-6D5K,False,True
4,NCIH1339-311Cas9_RepA_p6_Avana-3,SC-000921.AV01,21,Avana-3,A,ACH-000921,False,2DS,Avana,MC-000921-P628,False,True


In [12]:
seqmap['Library'].unique()

array(['Avana', 'Humagne-CD', 'KY'], dtype=object)

In [13]:
print('Days: ',seqmap['Days'].unique())
print('pDNA Batch: ',seqmap['pDNABatch'].unique())
print('SCreen Type: ',seqmap['ScreenType'].unique())
print('Library: ',seqmap['Library'].unique())
print('DrugTreated: ',seqmap['DrugTreated'].unique())
print(seqmap['ExcludeFromCRISPRCombined'].unique())


Days:  [21  0 14 15 22]
pDNA Batch:  ['Avana-4' 'Avana-3' 'Avana-2' 'CD-HiSeq' 'CD-NovaSeq' 'KY-1' 'KY-2']
SCreen Type:  ['2DS' 'pDNA']
Library:  ['Avana' 'Humagne-CD' 'KY']
DrugTreated:  [False nan]
[False]


In [14]:
# seqmap[seqmap['SequenceID'].str.startswith('Colo699-311cas9')]
seqmap[seqmap['ScreenType']=='pDNA']

,SequenceID,ScreenID,Days,pDNABatch,Replicate,ModelID,ExcludeFromCRISPRCombined,ScreenType,Library,ModelConditionID,DrugTreated,PassesQC
2281,pDNA_batch_Avana-4,pDNA,0,Avana-4,NaN,pDNA,False,pDNA,Avana,pDNA,NaN,True
2282,pDNA_batch_Avana-3,pDNA,0,Avana-3,NaN,pDNA,False,pDNA,Avana,pDNA,NaN,True
2283,pDNA_batch_Avana-2,pDNA,0,Avana-2,NaN,pDNA,False,pDNA,Avana,pDNA,NaN,True
2348,pDNA_CD-NovaSeq_0,pDNA,0,CD-NovaSeq,NaN,pDNA,False,pDNA,Humagne-CD,pDNA,NaN,True
2349,pDNA_CD-HiSeq_0,pDNA,0,CD-HiSeq,NaN,pDNA,False,pDNA,Humagne-CD,pDNA,NaN,True
3236,pDNA_batch_KY-1,pDNA,0,KY-1,NaN,pDNA,False,pDNA,KY,pDNA,NaN,True
3237,pDNA_batch_KY-2,pDNA,0,KY-2,NaN,pDNA,False,pDNA,KY,pDNA,NaN,True


In [15]:
print(seqmap[seqmap["PassesQC"] == True].shape)
print (seqmap.shape)
print(seqmap["pDNABatch"].unique())

(3094, 12)
(3238, 12)
['Avana-4' 'Avana-3' 'Avana-2' 'CD-HiSeq' 'CD-NovaSeq' 'KY-1' 'KY-2']


## **<span style="color:orangered;">Step 3 — Cleaning the data and</span>**

In [16]:
seqmap[seqmap['SequenceID'].isna()]

,SequenceID,ScreenID,Days,pDNABatch,Replicate,ModelID,ExcludeFromCRISPRCombined,ScreenType,Library,ModelConditionID,DrugTreated,PassesQC


In [17]:
screen_pass = seqmap[seqmap['SequenceID'].notna() & 
                  (seqmap['ScreenType'] == '2DS') &
                  (seqmap['ExcludeFromCRISPRCombined'] == False) &
                  (seqmap['Library']=='Avana') & 
                  seqmap['PassesQC'] == True]

pdna_pass = seqmap[seqmap['SequenceID'].notna() & 
    (seqmap['ScreenType']== 'pDNA') &
    seqmap['pDNABatch'].notna()]

print('seqmap shape:', seqmap.shape)
print('screen_pass shape:', screen_pass.shape)
print(f"pdna_pass shape: {pdna_pass.shape}")


seqmap shape: (3238, 12)
screen_pass shape: (2148, 12)
pdna_pass shape: (7, 12)


In [18]:
pdna_pass

,SequenceID,ScreenID,Days,pDNABatch,Replicate,ModelID,ExcludeFromCRISPRCombined,ScreenType,Library,ModelConditionID,DrugTreated,PassesQC
2281,pDNA_batch_Avana-4,pDNA,0,Avana-4,NaN,pDNA,False,pDNA,Avana,pDNA,NaN,True
2282,pDNA_batch_Avana-3,pDNA,0,Avana-3,NaN,pDNA,False,pDNA,Avana,pDNA,NaN,True
2283,pDNA_batch_Avana-2,pDNA,0,Avana-2,NaN,pDNA,False,pDNA,Avana,pDNA,NaN,True
2348,pDNA_CD-NovaSeq_0,pDNA,0,CD-NovaSeq,NaN,pDNA,False,pDNA,Humagne-CD,pDNA,NaN,True
2349,pDNA_CD-HiSeq_0,pDNA,0,CD-HiSeq,NaN,pDNA,False,pDNA,Humagne-CD,pDNA,NaN,True
3236,pDNA_batch_KY-1,pDNA,0,KY-1,NaN,pDNA,False,pDNA,KY,pDNA,NaN,True
3237,pDNA_batch_KY-2,pDNA,0,KY-2,NaN,pDNA,False,pDNA,KY,pDNA,NaN,True


In [33]:
pdna_pass_avana = pdna_pass[pdna_pass['pDNABatch'].str.startswith('Avana', na=False)]
pdna_pass_avana

,SequenceID,ScreenID,Days,pDNABatch,Replicate,ModelID,ExcludeFromCRISPRCombined,ScreenType,Library,ModelConditionID,DrugTreated,PassesQC
2281,pDNA_batch_Avana-4,pDNA,0,Avana-4,NaN,pDNA,False,pDNA,Avana,pDNA,NaN,True
2282,pDNA_batch_Avana-3,pDNA,0,Avana-3,NaN,pDNA,False,pDNA,Avana,pDNA,NaN,True
2283,pDNA_batch_Avana-2,pDNA,0,Avana-2,NaN,pDNA,False,pDNA,Avana,pDNA,NaN,True


In [19]:
screen_pass.head()

,SequenceID,ScreenID,Days,pDNABatch,Replicate,ModelID,ExcludeFromCRISPRCombined,ScreenType,Library,ModelConditionID,DrugTreated,PassesQC
0,Colo699-311cas9-RepA-p6_Avana-4,SC-001041.AV01,21,Avana-4,A,ACH-001041,False,2DS,Avana,MC-001041-CkXW,False,True
1,Colo699-311cas9-RepB-p6_Avana-4,SC-001041.AV01,21,Avana-4,B,ACH-001041,False,2DS,Avana,MC-001041-CkXW,False,True
2,HSB-2-311cas9_RepA_p4_Avana-3,SC-001737.AV01,21,Avana-3,A,ACH-001737,False,2DS,Avana,MC-001737-6D5K,False,True
3,HSB-2-311cas9_RepB_p4_Avana-3,SC-001737.AV01,21,Avana-3,B,ACH-001737,False,2DS,Avana,MC-001737-6D5K,False,True
4,NCIH1339-311Cas9_RepA_p6_Avana-3,SC-000921.AV01,21,Avana-3,A,ACH-000921,False,2DS,Avana,MC-000921-P628,False,True


### **<span style="color:#009900;">Step 3.1 — Understand your raw readcount matrix orientation</span>**

Do column names in "raw" match rep_map['SequenceID']?

In [20]:
print(f"set results: {len(set(screen_pass['SequenceID']) & set(raw.columns))}") # SequenceIDs that exist in BOTH screen_pass and raw it is similar to A & B (See the next Cell) 
print(f"QC-passing screen SequenceIDs: {len(screen_pass['SequenceID'].unique())}")
print(f"raw columns: {len(raw.columns)}")

set results: 2148
QC-passing screen SequenceIDs: 2148
raw columns: 2285


**nol:** Columns in raw that are NOT in screen_pass. This will include:  
    – pDNA columns ✅.  
    – guide column (Unnamed: 0) ✅.  
    – any other metadata columns ✅.  
    – QC-failed replicates ✅.  

In [21]:
A = set(screen_pass['SequenceID'])
B = set(raw.columns)

nol = B - A # nol = non_overlap
overlap = A & B # SequenceIDs that exist in BOTH screen_pass and raw
missing = A - B # missing = SequenceIDs that are in screen_pass but NOT in raw (Ideally len(missing) should be 0)

print(f"Non-overlapping SequenceIDs: {len(nol)}")
print(f"Overlapping SequenceIDs: {len(overlap)}")
print(f"Missing SequenceIDs: {len(missing)}")

Non-overlapping SequenceIDs: 137
Overlapping SequenceIDs: 2148
Missing SequenceIDs: 0


In [ ]:
list(nol)[:20]

In [23]:
nol_df = seqmap[
    seqmap['SequenceID'].isin(nol)
]

nol_df.head(5)

,SequenceID,ScreenID,Days,pDNABatch,Replicate,ModelID,ExcludeFromCRISPRCombined,ScreenType,Library,ModelConditionID,DrugTreated,PassesQC
90,MEG01-311cas9_RepA_p6_Avana-3,SC-000072.AV01,21,Avana-3,A,ACH-000072,False,2DS,Avana,MC-000072-l5Ss,False,False
91,MEG01-311cas9_RepB_p6_Avana-3,SC-000072.AV01,21,Avana-3,B,ACH-000072,False,2DS,Avana,MC-000072-l5Ss,False,False
103,BDCM-311Cas9_RepA_p6_Avana-3,SC-000080.AV01,21,Avana-3,A,ACH-000080,False,2DS,Avana,MC-000080-flxZ,False,False
104,BDCM-311Cas9_RepB_p6_Avana-3,SC-000080.AV01,21,Avana-3,B,ACH-000080,False,2DS,Avana,MC-000080-flxZ,False,False
141,SIGM5-311CAS9_RepA_p5_Avana-3,SC-000112.AV01,21,Avana-3,A,ACH-000112,False,2DS,Avana,MC-000112-bjRJ,False,False


In [24]:
print('Number of rows in screen_pass:', screen_pass.shape[0])
print('Number of columns in raw:', raw.shape[1])

Number of rows in screen_pass: 2148
Number of columns in raw: 2285


### **<span style="color:#009900;">Step 3.2 — Create a “screen counts” matrix and a “pDNA counts” matrix</span>**

**Action:**  
Split the raw matrix into two views:     
– counts for QC-passing screen replicates  
– counts for pDNA replicates (grouped by pDNA batch)   

**Hints:**  
– Use your replicate lists from screen_pass and pdna_pass     
– This is where you avoid mixing baselines and screens by accident     

**Code moves:**
– screen_cols = list(screen_pass[replicate_id_col])     
– pdna_cols = list(pdna_pass[replicate_id_col])    
– Subset the raw dataframe to those columns (+ guide id)   

**Sanity checks:**    
– No missing replicate columns (if some are missing, print which ones are missing — it will tell you if naming differs)    

In [28]:
screen_seqids_pass = screen_pass['SequenceID'].unique()
len(screen_seqids_pass)
screen_seqids_pass

array(['Colo699-311cas9-RepA-p6_Avana-4',
       'Colo699-311cas9-RepB-p6_Avana-4', 'HSB-2-311cas9_RepA_p4_Avana-3',
       ..., 'UPMM3-311cas9-RepA-p4_Avana-4',
       'UACC3199-311cas9-RepA-p4_Avana-4',
       'UACC3199-311cas9-RepB-p4_Avana-4'], shape=(2148,), dtype=object)

In [30]:
raw_screen_pass = raw[['sgRNA_seq'] + list(screen_seqids_pass)]
raw_screen_pass.head()

,sgRNA_seq,Colo699-311cas9-RepA-p6_Avana-4,Colo699-311cas9-RepB-p6_Avana-4,HSB-2-311cas9_RepA_p4_Avana-3,HSB-2-311cas9_RepB_p4_Avana-3,NCIH1339-311Cas9_RepA_p6_Avana-3,NCIH1339-311Cas9_RepB_p6_Avana-3,HEL-311Cas9_RepA_p4_Avana-3,HEL-311Cas9_RepB_p4_Avana-3,HEL9217-311Cas9_RepA_p6_Avana-3,...,SCCH196-311Cas9-RepB-P6_Avana-4,SNU977-311Cas9-AVANA-RepA-p6_Avana-4,SNU977-311Cas9-AVANA-RepB-p6_Avana-4,HMMME-311cas9-RepA-p6_Avana-4,HMMME-311cas9-RepB-p6_Avana-4,UACC3133-311Cas9-RepB-P6_Avana-4,LU139-311cas9-RepB-p5_Avana-4,UPMM3-311cas9-RepA-p4_Avana-4,UACC3199-311cas9-RepA-p4_Avana-4,UACC3199-311cas9-RepB-p4_Avana-4
0,AAAAAAATCCAGCAATGCAG,463.0,521.0,514.0,776.0,474.0,329.0,214.0,254.0,510.0,...,411.0,246.0,287.0,366.0,450.0,460.0,419.0,275.0,528.0,546.0
1,AAAAAACCCGTAGATAGCCT,570.0,596.0,581.0,756.0,398.0,281.0,218.0,270.0,603.0,...,715.0,325.0,485.0,884.0,741.0,406.0,577.0,385.0,733.0,608.0
2,AAAAAAGAAGAAAAAACCAG,93.0,79.0,135.0,244.0,85.0,45.0,106.0,118.0,171.0,...,90.0,42.0,110.0,49.0,52.0,171.0,45.0,100.0,149.0,127.0
3,AAAAAAGCTCAAGAAGGAGG,168.0,150.0,153.0,268.0,134.0,161.0,69.0,105.0,258.0,...,172.0,99.0,148.0,270.0,205.0,209.0,99.0,101.0,217.0,181.0
4,AAAAAAGGCTGTAAAAGCGT,197.0,264.0,276.0,270.0,245.0,129.0,181.0,120.0,345.0,...,223.0,92.0,214.0,325.0,352.0,267.0,197.0,206.0,379.0,347.0


In [27]:
type(sequenceids_pass)

numpy.ndarray

In [34]:
pdna_seqids_pass = pdna_pass_avana['SequenceID'].unique()
len(pdna_seqids_pass)
pdna_seqids_pass

array(['pDNA_batch_Avana-4', 'pDNA_batch_Avana-3', 'pDNA_batch_Avana-2'],
      dtype=object)

In [36]:
raw_pdna_avana_pass = raw[['sgRNA_seq'] + list(pdna_seqids_pass)]
raw_pdna_avana_pass.head()

,sgRNA_seq,pDNA_batch_Avana-4,pDNA_batch_Avana-3,pDNA_batch_Avana-2
0,AAAAAAATCCAGCAATGCAG,2165.0,6327.0,190.5
1,AAAAAACCCGTAGATAGCCT,3127.5,9322.0,289.0
2,AAAAAAGAAGAAAAAACCAG,517.0,2030.5,41.5
3,AAAAAAGCTCAAGAAGGAGG,918.0,3333.5,82.5
4,AAAAAAGGCTGTAAAAGCGT,1184.0,4038.0,117.5


In [ ]:
screen_pass.to_csv(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-nima.fakouri@gmail.com/My Drive/informatics/AI projects/crispr_ai_ml/data/processed/phase3_inputs/screen_metadata_pass.csv"), index=False)
pdna_pass.to_csv(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-nima.fakouri@gmail.com/My Drive/informatics/AI projects/crispr_ai_ml/data/processed/phase3_inputs/pdna_metadata_all.csv"), index=False)
pdna_pass_avana.to_csv(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-nima.fakouri@gmail.com/My Drive/informatics/AI projects/crispr_ai_ml/data/processed/phase3_inputs/pdna_metadata_avana.csv"), index=False)
raw_screen_pass.to_csv(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-nima.fakouri@gmail.com/My Drive/informatics/AI projects/crispr_ai_ml/data/processed/phase3_inputs/raw_screen_counts_pass.csv"), index=False)
raw_pdna_avana_pass.to_csv(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-nima.fakouri@gmail.com/My Drive/informatics/AI projects/crispr_ai_ml/data/processed/phase3_inputs/raw_pdna_avana_counts_pass.csv"), index=False)